A PySpark notebook was used because it provided a clear, code-based and reproducible transformation from Silver to Gold.

In [1]:
from pyspark.sql import functions as F

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 3, Finished, Available, Finished, False)

In [2]:
# Display all tables available in the attached Lakehouse

spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 4, Finished, Available, Finished, False)

+------------------------------------------------------+------------------------+-----------+
|namespace                                             |tableName               |isTemporary|
+------------------------------------------------------+------------------------+-----------+
|`Project NHL_Kee_win_factors`.nhl_lakehouse_silver.dbo|silver_game             |false      |
|`Project NHL_Kee_win_factors`.nhl_lakehouse_silver.dbo|silver_game_goalie_stats|false      |
|`Project NHL_Kee_win_factors`.nhl_lakehouse_silver.dbo|silver_game_skater_stats|false      |
|`Project NHL_Kee_win_factors`.nhl_lakehouse_silver.dbo|silver_game_teams_stats |false      |
|`Project NHL_Kee_win_factors`.nhl_lakehouse_silver.dbo|silver_player_info      |false      |
|`Project NHL_Kee_win_factors`.nhl_lakehouse_silver.dbo|silver_team_info        |false      |
+------------------------------------------------------+------------------------+-----------+



##### Inspect the six Silver tables

In [3]:
silver_tables = [
    "silver_game",
    "silver_game_teams_stats",
    "silver_team_info",
    "silver_game_skater_stats",
    "silver_player_info",
    "silver_game_goalie_stats"
]

for table_name in silver_tables:
    df = spark.table(table_name)

    print("=" * 70)
    print(f"TABLE: {table_name}")
    print(f"ROW COUNT: {df.count():,}")
    print(f"COLUMN COUNT: {len(df.columns)}")
    print("COLUMNS:")
    print(df.columns)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 5, Finished, Available, Finished, False)

TABLE: silver_game
ROW COUNT: 23,735
COLUMN COUNT: 14
COLUMNS:
['game_id', 'season', 'type', 'date_time_gmt', 'away_team_id', 'home_team_id', 'away_goals', 'home_goals', 'outcome', 'home_rink_side_start', 'venue', 'venue_time_zone_id', 'venue_time_zone_offset', 'venue_time_zone_tz']
TABLE: silver_game_teams_stats
ROW COUNT: 47,462
COLUMN COUNT: 17
COLUMNS:
['game_id', 'team_id', 'home_or_away', 'won', 'settled_in', 'head_coach', 'goals', 'shots', 'hits', 'pim', 'power_play_opportunities', 'power_play_goals', 'face_off_win_percentage', 'giveaways', 'takeaways', 'blocked', 'start_rink_side']
TABLE: silver_team_info
ROW COUNT: 33
COLUMN COUNT: 6
COLUMNS:
['team_id', 'franchise_id', 'short_name', 'team_name', 'abbreviation', 'link']
TABLE: silver_game_skater_stats
ROW COUNT: 853,404
COLUMN COUNT: 22
COLUMNS:
['game_id', 'player_id', 'team_id', 'time_on_ice', 'assists', 'goals', 'shots', 'hits', 'power_play_goals', 'power_play_assists', 'penalty_minutes', 'face_off_wins', 'face_off_taken', 

The six Silver tables operate at different levels of detail. Game and team-statistics data can be represented at game or team-game level, while skater and goalie tables contain multiple player records per team-game. Directly joining them would create a many-to-many row multiplication problem. Player statistics therefore need to be aggregated to the team-game level before constructing the Gold analytical table.

The game table contains 23,735 games, giving a theoretical expectation of 47,470 team-game records. The game-team-statistics table contains 47,462 records, a difference of eight rows. Key uniqueness and per-game team-record counts must be validated before Gold-table construction.

##### Validate table grains and keys

In [4]:
key_definitions = [
    ("silver_game", ["game_id"]),
    ("silver_game_teams_stats", ["game_id", "team_id"]),
    ("silver_team_info", ["team_id"]),
    ("silver_game_skater_stats", ["game_id", "player_id"]),
    ("silver_player_info", ["player_id"]),
    ("silver_game_goalie_stats", ["game_id", "player_id"])
]

validation_results = []

for table_name, key_columns in key_definitions:
    df = spark.table(table_name)

    total_rows = df.count()
    distinct_keys = df.select(key_columns).distinct().count()
    duplicate_rows = total_rows - distinct_keys

    null_condition = None

    for column_name in key_columns:
        condition = F.col(column_name).isNull()
        null_condition = (
            condition if null_condition is None
            else null_condition | condition
        )

    null_key_rows = df.filter(null_condition).count()

    validation_results.append(
        (
            table_name,
            ", ".join(key_columns),
            total_rows,
            distinct_keys,
            duplicate_rows,
            null_key_rows
        )
    )

key_validation_df = spark.createDataFrame(
    validation_results,
    [
        "table_name",
        "expected_key",
        "total_rows",
        "distinct_keys",
        "duplicate_rows",
        "null_key_rows"
    ]
)

display(key_validation_df)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 264ebf41-5194-4909-8b56-aa54b8af197a)

##### Check team-stat rows per game

Each NHL game should normally have exactly two team-stat rows.

In [5]:
game_df = spark.table("silver_game")
team_stats_df = spark.table("silver_game_teams_stats")

# Count team-stat rows associated with every game
team_rows_per_game = (
    team_stats_df
    .groupBy("game_id")
    .agg(F.count("*").alias("team_stat_rows"))
)

# Retain every game, including games with no team-stat records
game_team_coverage = (
    game_df
    .select("game_id", "season", "date_time_gmt",
            "away_team_id", "home_team_id",
            "away_goals", "home_goals")
    .join(team_rows_per_game, on="game_id", how="left")
    .fillna({"team_stat_rows": 0})
)

# Show how many games have 0, 1, 2 or more team-stat rows
display(
    game_team_coverage
    .groupBy("team_stat_rows")
    .agg(F.count("*").alias("number_of_games"))
    .orderBy("team_stat_rows")
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 644e6766-e31a-4434-beed-81886724a19b)

23,731 games have exactly two team-stat rows.

Four games have no matching team-stat records. Those four games account for all eight missing team-game rows: 4 × 2 = 8.

##### Identify the four games

In [6]:
display(
    game_team_coverage
    .filter(F.col("team_stat_rows") != 2)
    .orderBy("date_time_gmt")
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 99ef6944-2db6-4108-961e-85af005872d6)

Four records in silver_game represented conditional playoff fixtures that were never played. Each had a 0–0 score and no corresponding team, skater or goalie performance statistics.

They were excluded from the analytical Gold dataset, leaving 23,731 valid games and 47,462 team-game observations.

##### Validate relationships between tables

In [7]:
game_df = spark.table("silver_game")
team_stats_df = spark.table("silver_game_teams_stats")
team_info_df = spark.table("silver_team_info")
skater_df = spark.table("silver_game_skater_stats")
player_info_df = spark.table("silver_player_info")
goalie_df = spark.table("silver_game_goalie_stats")

relationship_checks = [
    (
        "Team stats without matching game",
        team_stats_df.select("game_id").distinct()
        .join(game_df.select("game_id"), "game_id", "left_anti")
        .count()
    ),
    (
        "Team stats without matching team",
        team_stats_df.select("team_id").distinct()
        .join(team_info_df.select("team_id"), "team_id", "left_anti")
        .count()
    ),
    (
        "Skater stats without matching game",
        skater_df.select("game_id").distinct()
        .join(game_df.select("game_id"), "game_id", "left_anti")
        .count()
    ),
    (
        "Skater stats without matching team",
        skater_df.select("team_id").distinct()
        .join(team_info_df.select("team_id"), "team_id", "left_anti")
        .count()
    ),
    (
        "Skater stats without matching player",
        skater_df.select("player_id").distinct()
        .join(player_info_df.select("player_id"), "player_id", "left_anti")
        .count()
    ),
    (
        "Goalie stats without matching game",
        goalie_df.select("game_id").distinct()
        .join(game_df.select("game_id"), "game_id", "left_anti")
        .count()
    ),
    (
        "Goalie stats without matching team",
        goalie_df.select("team_id").distinct()
        .join(team_info_df.select("team_id"), "team_id", "left_anti")
        .count()
    ),
    (
        "Goalie stats without matching player",
        goalie_df.select("player_id").distinct()
        .join(player_info_df.select("player_id"), "player_id", "left_anti")
        .count()
    )
]

relationship_validation_df = spark.createDataFrame(
    relationship_checks,
    ["relationship_check", "unmatched_records"]
)

display(relationship_validation_df)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0b1ec3f5-9352-4188-9306-0cf977067792)

##### Identify the missing team IDs

In [8]:
valid_team_ids = team_info_df.select("team_id").distinct()

missing_team_ids = (
    team_stats_df
    .select("team_id")
    .union(skater_df.select("team_id"))
    .union(goalie_df.select("team_id"))
    .distinct()
    .join(valid_team_ids, on="team_id", how="left_anti")
)

display(missing_team_ids.orderBy("team_id"))

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 35e3933c-ff93-486d-97be-b1adb7a8f8a1)

In [9]:
# No. of records each missing team has in the three statistics tables

missing_team_counts = (
    missing_team_ids.alias("m")
    .join(
        team_stats_df
        .groupBy("team_id")
        .agg(F.count("*").alias("team_stat_rows")),
        on="team_id",
        how="left"
    )
    .join(
        skater_df
        .groupBy("team_id")
        .agg(F.count("*").alias("skater_rows")),
        on="team_id",
        how="left"
    )
    .join(
        goalie_df
        .groupBy("team_id")
        .agg(F.count("*").alias("goalie_rows")),
        on="team_id",
        how="left"
    )
    .fillna(0)
)

display(missing_team_counts.orderBy("team_id"))

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bb378795-6ade-4af5-acc0-b1f7a8a5a266)

##### Inspect their games

In [10]:
# To confirm the game type and dates of the missing teams' games

missing_ids = [87, 88, 89, 90]

missing_team_games = (
    team_stats_df
    .filter(F.col("team_id").isin(missing_ids))
    .select(
        "game_id",
        "team_id",
        "home_or_away",
        "won",
        "settled_in",
        "goals",
        "shots"
    )
    .join(
        game_df.select(
            "game_id",
            "season",
            "type",
            "date_time_gmt",
            "away_team_id",
            "home_team_id",
            "away_goals",
            "home_goals"
        ),
        on="game_id",
        how="left"
    )
    .orderBy("date_time_gmt", "team_id")
)

display(missing_team_games)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 10eb87f4-77d1-44e5-a07a-4aa8a7e14763)

These are All-Star games.

type = "A"

IDs 87–90 play only against one another.

The games occurred during the NHL All-Star weekends.

Scores and roster sizes differ noticeably from ordinary competitive games.

These games are excluded because the analysis concerns factors associated with winning normal NHL games—not exhibition All-Star games.

Final exclusions so far:

4 unplayed conditional playoff -- No performance statistics

5 All-Star exhibition games -- Different format and competitive context

Balance:

23,726 competitive games

47,452 team-game rows

##### Build the Gold base dataframe

In [11]:
from pyspark.sql import functions as F

# Retain competitive regular-season and playoff games
valid_game_df = (
    game_df
    .filter(F.col("type").isin("R", "P"))
)

gold_base_df = (
    team_stats_df.alias("ts")
    .join(
        valid_game_df.alias("g"),
        F.col("ts.game_id") == F.col("g.game_id"),
        "inner"
    )
    .join(
        team_info_df.alias("ti"),
        F.col("ts.team_id") == F.col("ti.team_id"),
        "left"
    )
    .select(
        F.col("ts.game_id").alias("game_id"),
        F.col("g.season").alias("season"),
        F.col("g.type").alias("game_type"),
        F.col("g.date_time_gmt").alias("game_date_time_gmt"),

        F.col("ts.team_id").alias("team_id"),
        F.concat_ws(
            " ",
            F.col("ti.short_name"),
            F.col("ti.team_name")
        ).alias("team_name"),
        F.col("ti.abbreviation").alias("team_abbreviation"),

        F.col("ts.home_or_away").alias("home_or_away"),
        F.col("ts.won").alias("is_win"),
        F.col("ts.settled_in").alias("settled_in"),
        F.col("ts.head_coach").alias("head_coach"),

        F.when(
            F.col("ts.home_or_away") == "home",
            F.col("g.away_team_id")
        ).otherwise(
            F.col("g.home_team_id")
        ).alias("opponent_team_id"),

        F.when(
            F.col("ts.home_or_away") == "home",
            F.col("g.away_goals")
        ).otherwise(
            F.col("g.home_goals")
        ).alias("opponent_goals"),

        F.col("ts.goals").alias("goals"),
        F.col("ts.shots").alias("shots"),
        F.col("ts.hits").alias("hits"),
        F.col("ts.pim").alias("penalty_minutes"),
        F.col("ts.power_play_opportunities"),
        F.col("ts.power_play_goals"),
        F.col("ts.face_off_win_percentage"),
        F.col("ts.giveaways"),
        F.col("ts.takeaways"),
        F.col("ts.blocked"),
        F.col("ts.start_rink_side")
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 13, Finished, Available, Finished, False)

In [12]:
# Validate the base

gold_base_validation = gold_base_df.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("game_id").alias("distinct_games"),
    F.countDistinct(
        F.struct("game_id", "team_id")
    ).alias("distinct_team_games"),
    F.sum(
        F.when(F.col("team_name").isNull(), 1).otherwise(0)
    ).alias("missing_team_names")
)

display(gold_base_validation)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2fc7a0c0-cf8c-49e6-85ca-8b2a0ded2e4d)

##### Aggregate skater performance

To reduce the 853,404 skater records to one row per team per game before joining them to gold_base_df.

In [13]:
# Add player position and calculate player-level points and time on ice

skater_enriched_df = (
    skater_df.alias("s")
    .join(
        valid_game_df.select("game_id").alias("vg"),
        on="game_id",
        how="inner"
    )
    .join(
        player_info_df
        .select("player_id", "primary_position")
        .alias("p"),
        on="player_id",
        how="left"
    )
    .withColumn(
        "player_points",
        F.coalesce(F.col("goals"), F.lit(0)) +
        F.coalesce(F.col("assists"), F.lit(0))
    )
    .withColumn(
        "time_on_ice_seconds",
        F.when(
            F.col("time_on_ice").isNotNull(),
            F.split(F.col("time_on_ice"), ":").getItem(0).cast("int") * 60 +
            F.split(F.col("time_on_ice"), ":").getItem(1).cast("int")
        )
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 15, Finished, Available, Finished, False)

In [14]:
skater_team_game_df = (
    skater_enriched_df
    .groupBy("game_id", "team_id")
    .agg(
        F.countDistinct("player_id").alias("skaters_used"),

        F.sum("goals").alias("skater_goals"),
        F.sum("assists").alias("skater_assists"),
        F.sum("shots").alias("skater_shots"),
        F.sum("hits").alias("skater_hits"),
        F.sum("blocked").alias("skater_blocked"),
        F.sum("takeaways").alias("skater_takeaways"),
        F.sum("giveaways").alias("skater_giveaways"),
        F.sum("penalty_minutes").alias("skater_penalty_minutes"),
        F.sum("plus_minus").alias("team_plus_minus"),

        F.sum("time_on_ice_seconds").alias("total_skater_toi_seconds"),
        F.avg("time_on_ice_seconds").alias("average_skater_toi_seconds"),

        F.sum(
            F.when(F.col("goals") > 0, 1).otherwise(0)
        ).alias("goal_scorers"),

        F.sum(
            F.when(F.col("player_points") > 0, 1).otherwise(0)
        ).alias("point_contributors"),

        F.max("player_points").alias("top_player_points"),

        F.sum(
            F.when(F.col("primary_position") == "D", 1).otherwise(0)
        ).alias("defencemen_used"),

        F.sum(
            F.when(
                F.col("primary_position").isin("C", "L", "R"),
                1
            ).otherwise(0)
        ).alias("forwards_used")
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 16, Finished, Available, Finished, False)

In [15]:
# Validate its grain

display(
    skater_team_game_df.agg(
        F.count("*").alias("total_team_game_rows"),
        F.countDistinct("game_id").alias("distinct_games"),
        F.countDistinct(
            F.struct("game_id", "team_id")
        ).alias("distinct_team_games")
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7e39811e-d3ab-42c6-92f1-a68d3e1ba7e4)

Gold base: 47,452 team-game rows

Skater aggregate: 47,432 team-game rows

Difference: 20 team-game rows

Gold games: 23,726

Skater games: 23,716

Difference: 10 games

##### Identify the 10 games

In [16]:
missing_skater_team_games_df = (
    gold_base_df
    .select(
        "game_id",
        "season",
        "game_type",
        "game_date_time_gmt",
        "team_id",
        "team_name",
        "home_or_away",
        "is_win",
        "goals",
        "opponent_goals"
    )
    .join(
        skater_team_game_df.select("game_id", "team_id"),
        on=["game_id", "team_id"],
        how="left_anti"
    )
    .orderBy("game_date_time_gmt", "game_id", "home_or_away")
)

display(missing_skater_team_games_df)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3cfd2490-dace-45bd-801f-54b82f518bf7)

In [17]:
display(
    missing_skater_team_games_df
    .groupBy("season", "game_type")
    .agg(
        F.countDistinct("game_id").alias("games"),
        F.count("*").alias("team_game_rows")
    )
    .orderBy("season", "game_type")
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ea959faf-cb04-40b0-80b6-046c444ecbb9)

##### Display the 10 affected games

In [18]:
missing_skater_games_df = (
    missing_skater_team_games_df
    .groupBy(
        "game_id",
        "season",
        "game_type",
        "game_date_time_gmt"
    )
    .agg(
        F.max(
            F.when(
                F.col("home_or_away") == "home",
                F.col("team_name")
            )
        ).alias("home_team"),

        F.max(
            F.when(
                F.col("home_or_away") == "away",
                F.col("team_name")
            )
        ).alias("away_team"),

        F.max(
            F.when(
                F.col("home_or_away") == "home",
                F.col("goals")
            )
        ).alias("home_goals"),

        F.max(
            F.when(
                F.col("home_or_away") == "away",
                F.col("goals")
            )
        ).alias("away_goals")
    )
    .orderBy("game_date_time_gmt")
)

display(missing_skater_games_df)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, abac194a-2173-459b-92a8-a82f39944e51)

These 10 records are also unplayed conditional playoff fixtures.

All have: A 0–0 score, no skater statistics, dates corresponding to potential later games in playoff series that had already ended.

Combined with the earlier four fixtures, there are 14 unplayed games in silver_game.

<u>**Confirmed analytical population**</u>

Original game records:	23,735

Unplayed playoff fixtures:	14 (excluded)

All-Star exhibition games: 5 (excluded)

Competitive completed games retained:	23,716

Expected Gold grain: 47,432 team-game rows

##### Rebuild the Gold base dataframe

In [19]:
from pyspark.sql import functions as F

# Game IDs with recorded player performance
games_with_skater_data = (
    skater_df
    .select("game_id")
    .distinct()
)

# Retain regular-season and playoff games that have player statistics
valid_game_df = (
    game_df
    .filter(F.col("type").isin("R", "P"))
    .join(
        games_with_skater_data,
        on="game_id",
        how="inner"
    )
)

gold_base_df = (
    team_stats_df.alias("ts")
    .join(
        valid_game_df.alias("g"),
        F.col("ts.game_id") == F.col("g.game_id"),
        "inner"
    )
    .join(
        team_info_df.alias("ti"),
        F.col("ts.team_id") == F.col("ti.team_id"),
        "left"
    )
    .select(
        F.col("ts.game_id").alias("game_id"),
        F.col("g.season").alias("season"),
        F.col("g.type").alias("game_type"),
        F.col("g.date_time_gmt").alias("game_date_time_gmt"),

        F.col("ts.team_id").alias("team_id"),
        F.concat_ws(
            " ",
            F.col("ti.short_name"),
            F.col("ti.team_name")
        ).alias("team_name"),
        F.col("ti.abbreviation").alias("team_abbreviation"),

        F.col("ts.home_or_away").alias("home_or_away"),
        F.col("ts.won").alias("is_win"),
        F.col("ts.settled_in").alias("settled_in"),
        F.col("ts.head_coach").alias("head_coach"),

        F.when(
            F.col("ts.home_or_away") == "home",
            F.col("g.away_team_id")
        ).otherwise(
            F.col("g.home_team_id")
        ).alias("opponent_team_id"),

        F.when(
            F.col("ts.home_or_away") == "home",
            F.col("g.away_goals")
        ).otherwise(
            F.col("g.home_goals")
        ).alias("opponent_goals"),

        F.col("ts.goals").alias("goals"),
        F.col("ts.shots").alias("shots"),
        F.col("ts.hits").alias("hits"),
        F.col("ts.pim").alias("penalty_minutes"),
        F.col("ts.power_play_opportunities"),
        F.col("ts.power_play_goals"),
        F.col("ts.face_off_win_percentage"),
        F.col("ts.giveaways"),
        F.col("ts.takeaways"),
        F.col("ts.blocked"),
        F.col("ts.start_rink_side")
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 21, Finished, Available, Finished, False)

In [20]:
# Validate the base again

gold_base_validation = gold_base_df.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("game_id").alias("distinct_games"),
    F.countDistinct(
        F.struct("game_id", "team_id")
    ).alias("distinct_team_games"),
    F.sum(
        F.when(F.col("team_name").isNull(), 1).otherwise(0)
    ).alias("missing_team_names")
)

display(gold_base_validation)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4a34ef1a-926d-4376-b267-d278f44c59ab)

##### Aggregate skater performance (2nd attempt)

To reduce the 853,404 skater records to one row per team per game before joining them to gold_base_df.

In [21]:
# Add player position and calculate player-level points and time on ice

skater_enriched_df = (
    skater_df.alias("s")
    .join(
        valid_game_df.select("game_id").alias("vg"),
        on="game_id",
        how="inner"
    )
    .join(
        player_info_df
        .select("player_id", "primary_position")
        .alias("p"),
        on="player_id",
        how="left"
    )
    .withColumn(
        "player_points",
        F.coalesce(F.col("goals"), F.lit(0)) +
        F.coalesce(F.col("assists"), F.lit(0))
    )
    .withColumn(
        "time_on_ice_seconds",
        F.when(
            F.col("time_on_ice").isNotNull(),
            F.split(F.col("time_on_ice"), ":").getItem(0).cast("int") * 60 +
            F.split(F.col("time_on_ice"), ":").getItem(1).cast("int")
        )
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 23, Finished, Available, Finished, False)

In [22]:
skater_team_game_df = (
    skater_enriched_df
    .groupBy("game_id", "team_id")
    .agg(
        F.countDistinct("player_id").alias("skaters_used"),

        F.sum("goals").alias("skater_goals"),
        F.sum("assists").alias("skater_assists"),
        F.sum("shots").alias("skater_shots"),
        F.sum("hits").alias("skater_hits"),
        F.sum("blocked").alias("skater_blocked"),
        F.sum("takeaways").alias("skater_takeaways"),
        F.sum("giveaways").alias("skater_giveaways"),
        F.sum("penalty_minutes").alias("skater_penalty_minutes"),
        F.sum("plus_minus").alias("team_plus_minus"),

        F.sum("time_on_ice_seconds").alias("total_skater_toi_seconds"),
        F.avg("time_on_ice_seconds").alias("average_skater_toi_seconds"),

        F.sum(
            F.when(F.col("goals") > 0, 1).otherwise(0)
        ).alias("goal_scorers"),

        F.sum(
            F.when(F.col("player_points") > 0, 1).otherwise(0)
        ).alias("point_contributors"),

        F.max("player_points").alias("top_player_points"),

        F.sum(
            F.when(F.col("primary_position") == "D", 1).otherwise(0)
        ).alias("defencemen_used"),

        F.sum(
            F.when(
                F.col("primary_position").isin("C", "L", "R"),
                1
            ).otherwise(0)
        ).alias("forwards_used")
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 24, Finished, Available, Finished, False)

In [23]:
# Validate its grain

display(
    skater_team_game_df.agg(
        F.count("*").alias("total_team_game_rows"),
        F.countDistinct("game_id").alias("distinct_games"),
        F.countDistinct(
            F.struct("game_id", "team_id")
        ).alias("distinct_team_games")
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a26eb0b3-fbd7-497c-8045-a1dff7d6dab9)

##### Aggregate goalie performance

The goalie table can contain more than one goalie for a team in a game, so it is necessary to aggregate it to the same team-game grain.

In [24]:
goalie_enriched_df = (
    goalie_df.alias("gk")
    .join(
        valid_game_df.select("game_id"),
        on="game_id",
        how="inner"
    )
    .withColumn(
        "goalie_toi_seconds",
        F.when(
            F.col("time_on_ice").isNotNull(),
            F.split(F.col("time_on_ice"), ":").getItem(0).cast("int") * 60 +
            F.split(F.col("time_on_ice"), ":").getItem(1).cast("int")
        )
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 26, Finished, Available, Finished, False)

In [25]:
# Aggregate the goalie statistics

goalie_team_game_df = (
    goalie_enriched_df
    .groupBy("game_id", "team_id")
    .agg(
        F.countDistinct("player_id").alias("goalies_used"),

        F.sum("shots").alias("shots_against"),
        F.sum("saves").alias("goalie_saves"),

        F.sum("power_play_saves").alias("power_play_saves"),
        F.sum("short_handed_saves").alias("short_handed_saves"),
        F.sum("even_saves").alias("even_strength_saves"),

        F.sum("power_play_shots_against")
            .alias("power_play_shots_against"),

        F.sum("short_handed_shots_against")
            .alias("short_handed_shots_against"),

        F.sum("even_shots_against")
            .alias("even_strength_shots_against"),

        F.sum("assists").alias("goalie_assists"),
        F.sum("penalty_minutes").alias("goalie_penalty_minutes"),
        F.sum("goalie_toi_seconds").alias("total_goalie_toi_seconds")
    )
    .withColumn(
        "team_save_percentage",
        F.when(
            F.col("shots_against") > 0,
            F.round(
                F.col("goalie_saves") /
                F.col("shots_against") * 100,
                2
            )
        )
    )
    .withColumn(
        "used_multiple_goalies",
        F.when(F.col("goalies_used") > 1, 1).otherwise(0)
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 27, Finished, Available, Finished, False)

In [26]:
# Validate the goalie aggregation

display(
    goalie_team_game_df.agg(
        F.count("*").alias("total_team_game_rows"),
        F.countDistinct("game_id").alias("distinct_games"),
        F.countDistinct(
            F.struct("game_id", "team_id")
        ).alias("distinct_team_games")
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b9d73e70-aaaa-4390-8561-57f2dbc172ff)

Gold base: 47,432 team-games

Goalie aggregate: 47,425 team-games

Difference: 7

Both cover 23,716 distinct games

##### Identify the missing goalie records

In [27]:
missing_goalie_team_games_df = (
    gold_base_df
    .select(
        "game_id",
        "season",
        "game_type",
        "game_date_time_gmt",
        "team_id",
        "team_name",
        "home_or_away",
        "is_win",
        "goals",
        "opponent_goals",
        "shots"
    )
    .join(
        goalie_team_game_df.select("game_id", "team_id"),
        on=["game_id", "team_id"],
        how="left_anti"
    )
    .orderBy("game_date_time_gmt")
)

display(missing_goalie_team_games_df)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6d79bfb7-b0b8-484a-b7f4-0f453539be9f)

In [28]:
# Show the available goalie record for the opposing side of those games
affected_game_ids_df = (
    missing_goalie_team_games_df
    .select("game_id")
    .distinct()
)

display(
    goalie_team_game_df
    .join(
        affected_game_ids_df,
        on="game_id",
        how="inner"
    )
    .orderBy("game_id", "team_id")
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 79358b55-2f03-4b44-a469-c60f10c12a2d)

These seven are valid completed regular-season games from 2003–04 to 2007–08.

They have valid scores and win results, team statistics, skater statistics and goalie data for the opposing team.

Only one team’s goalie record is missing in each game.

Therefore, all seven team-game rows will be retained but the goalie-derived fields will be left as null.

The missing goalie statistics should not be replaced with zero. Zero would incorrectly imply that the goalie faced no shots or made no saves.

##### Join the Gold components

Left joins are used so that the full analytical population is preserved.

In [29]:
opponent_info_df = (
    team_info_df
    .select(
        F.col("team_id").alias("opponent_team_id"),
        F.concat_ws(
            " ",
            F.col("short_name"),
            F.col("team_name")
        ).alias("opponent_team_name"),
        F.col("abbreviation").alias("opponent_abbreviation")
    )
)

gold_combined_df = (
    gold_base_df
    .join(
        skater_team_game_df,
        on=["game_id", "team_id"],
        how="left"
    )
    .join(
        goalie_team_game_df,
        on=["game_id", "team_id"],
        how="left"
    )
    .join(
        opponent_info_df,
        on="opponent_team_id",
        how="left"
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 31, Finished, Available, Finished, False)

In [30]:
# Validate that the joins did not multiply or remove rows

gold_combined_validation = gold_combined_df.agg(
    F.count("*").alias("total_rows"),

    F.countDistinct("game_id").alias("distinct_games"),

    F.countDistinct(
        F.struct("game_id", "team_id")
    ).alias("distinct_team_games"),

    F.sum(
        F.when(F.col("skaters_used").isNull(), 1).otherwise(0)
    ).alias("missing_skater_records"),

    F.sum(
        F.when(F.col("goalies_used").isNull(), 1).otherwise(0)
    ).alias("missing_goalie_records"),

    F.sum(
        F.when(
            F.col("opponent_team_name").isNull(),
            1
        ).otherwise(0)
    ).alias("missing_opponent_names")
)

display(gold_combined_validation)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 56259607-e3b5-4520-a77d-528291f2a019)

All 47,432 team-game rows are preserved, with only the seven known missing goalie records.

##### Add reliable opponent statistics

Some valid games had inaccurate 0–0 values in silver_game, so the current opponent_goals inherited from that table may be wrong.

Opponent statistics are derived from the opposing team row in silver_game_teams_stats, which is more reliable.

In [31]:
# Create an opponent lookup from gold_base_df

opponent_performance_df = (
    gold_base_df
    .select(
        "game_id",
        F.col("team_id").alias("opponent_team_id"),
        F.col("goals").alias("validated_opponent_goals"),
        F.col("shots").alias("opponent_shots"),
        F.col("hits").alias("opponent_hits"),
        F.col("penalty_minutes").alias("opponent_penalty_minutes"),
        F.col("power_play_opportunities")
            .alias("opponent_power_play_opportunities"),
        F.col("power_play_goals").alias("opponent_power_play_goals"),
        F.col("face_off_win_percentage")
            .alias("opponent_face_off_win_percentage"),
        F.col("giveaways").alias("opponent_giveaways"),
        F.col("takeaways").alias("opponent_takeaways"),
        F.col("blocked").alias("opponent_blocked")
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 33, Finished, Available, Finished, False)

In [32]:
# Replace the unreliable opponent-goals field and add the other opponent measures

gold_combined_df = (
    gold_combined_df
    .drop("opponent_goals")
    .join(
        opponent_performance_df,
        on=["game_id", "opponent_team_id"],
        how="left"
    )
    .withColumnRenamed(
        "validated_opponent_goals",
        "opponent_goals"
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 34, Finished, Available, Finished, False)

In [33]:
# Validate the opponent match

display(
    gold_combined_df.agg(
        F.count("*").alias("total_rows"),

        F.countDistinct(
            F.struct("game_id", "team_id")
        ).alias("distinct_team_games"),

        F.sum(
            F.when(
                F.col("opponent_goals").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_opponent_goals"),

        F.sum(
            F.when(
                F.col("opponent_shots").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_opponent_shots")
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5d5a4e99-c296-4de5-9a43-d0b885839b70)

##### Engineer the analytical features

In [34]:
gold_features_df = (
    gold_combined_df

    # Date and categorical indicators
    .withColumn(
        "game_date",
        F.to_date("game_date_time_gmt")
    )
    .withColumn(
        "win_flag",
        F.col("is_win").cast("int")
    )
    .withColumn(
        "home_flag",
        F.when(F.col("home_or_away") == "home", 1).otherwise(0)
    )
    .withColumn(
        "playoff_flag",
        F.when(F.col("game_type") == "P", 1).otherwise(0)
    )

    # Scoring and shooting
    .withColumn(
        "goal_difference",
        F.col("goals") - F.col("opponent_goals")
    )
    .withColumn(
        "shot_difference",
        F.col("shots") - F.col("opponent_shots")
    )
    .withColumn(
        "shot_share_percentage",
        F.when(
            (F.col("shots") + F.col("opponent_shots")) > 0,
            F.round(
                F.col("shots") /
                (F.col("shots") + F.col("opponent_shots")) * 100,
                2
            )
        )
    )
    .withColumn(
        "shooting_percentage",
        F.when(
            F.col("shots") > 0,
            F.round(F.col("goals") / F.col("shots") * 100, 2)
        )
    )

    # Special teams
    .withColumn(
        "power_play_efficiency",
        F.when(
            F.col("power_play_opportunities") > 0,
            F.round(
                F.col("power_play_goals") /
                F.col("power_play_opportunities") * 100,
                2
            )
        )
    )

    # Physical play and puck management
    .withColumn(
        "hit_difference",
        F.col("hits") - F.col("opponent_hits")
    )
    .withColumn(
        "penalty_minutes_difference",
        F.col("penalty_minutes") -
        F.col("opponent_penalty_minutes")
    )
    .withColumn(
        "faceoff_percentage_difference",
        F.round(
            F.col("face_off_win_percentage") -
            F.col("opponent_face_off_win_percentage"),
            2
        )
    )
    .withColumn(
        "takeaway_difference",
        F.col("takeaways") - F.col("opponent_takeaways")
    )
    .withColumn(
        "giveaway_difference",
        F.col("giveaways") - F.col("opponent_giveaways")
    )
    .withColumn(
        "blocked_shot_difference",
        F.col("blocked") - F.col("opponent_blocked")
    )
    .withColumn(
        "puck_management_balance",
        F.col("takeaways") - F.col("giveaways")
    )

    # Player contribution
    .withColumn(
        "points_per_scorer",
        F.when(
            F.col("point_contributors") > 0,
            F.round(
                (F.col("skater_goals") + F.col("skater_assists")) /
                F.col("point_contributors"),
                2
            )
        )
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 36, Finished, Available, Finished, False)

In [35]:
# Validate that feature engineering did not change the grain

display(
    gold_features_df.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("game_id").alias("distinct_games"),
        F.countDistinct(
            F.struct("game_id", "team_id")
        ).alias("distinct_team_games"),
        F.sum("win_flag").alias("winning_team_rows")
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c9cc2ef4-1dfd-469e-83a0-d804f4ad1fb3)

The lower number of winning rows indicates that some historical games ended in ties.

The dataset includes seasons before the NHL eliminated ties in 2005–06.

In a tied game, both team rows have is_win = false.

Therefore, the game contributes zero winning rows.

23,716 games − 23,088 winning rows = 628 tied games

##### Confirm the tied games

In [36]:
wins_per_game_df = (
    gold_features_df
    .groupBy("game_id")
    .agg(
        F.sum("win_flag").alias("winning_rows_in_game")
    )
)

display(
    wins_per_game_df
    .groupBy("winning_rows_in_game")
    .agg(F.count("*").alias("number_of_games"))
    .orderBy("winning_rows_in_game")
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2f47d36b-d852-444e-8eea-9a1ace64a9c7)

##### Create the result category

In [37]:
gold_final_df = (
    gold_features_df
    .join(
        wins_per_game_df,
        on="game_id",
        how="left"
    )
    .withColumn(
        "result_category",
        F.when(F.col("win_flag") == 1, "Win")
        .when(F.col("winning_rows_in_game") == 0, "Tie")
        .otherwise("Loss")
    )
    .withColumn(
        "win_loss_analysis_flag",
        F.when(
            F.col("winning_rows_in_game") == 1,
            1
        ).otherwise(0)
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 39, Finished, Available, Finished, False)

A separate win_loss_analysis_flag was created so tied games can be excluded from binary winner-versus-loser analysis without being deleted from the Gold table.

win_loss_analysis_flag

1: include in winner-versus-loser comparisons

0: historical tie; retain in Gold but exclude from comparisons

In [38]:
# Validate the categories

display(
    gold_final_df
    .groupBy("result_category", "win_loss_analysis_flag")
    .agg(
        F.count("*").alias("team_game_rows"),
        F.countDistinct("game_id").alias("distinct_games")
    )
    .orderBy("result_category")
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b4f45b2a-14fa-49bb-b86c-a8c6317f2b29)

##### Reconcile duplicated performance metrics

In [39]:
metric_reconciliation = gold_final_df.agg(
    F.sum(
        F.when(F.col("goals") != F.col("skater_goals"), 1).otherwise(0)
    ).alias("goal_mismatches"),

    F.sum(
        F.when(F.col("shots") != F.col("skater_shots"), 1).otherwise(0)
    ).alias("shot_mismatches"),

    F.sum(
        F.when(F.col("hits") != F.col("skater_hits"), 1).otherwise(0)
    ).alias("hit_mismatches"),

    F.sum(
        F.when(
            F.col("penalty_minutes") != F.col("skater_penalty_minutes"),
            1
        ).otherwise(0)
    ).alias("penalty_minute_mismatches"),

    F.sum(
        F.when(
            F.col("giveaways") != F.col("skater_giveaways"),
            1
        ).otherwise(0)
    ).alias("giveaway_mismatches"),

    F.sum(
        F.when(
            F.col("takeaways") != F.col("skater_takeaways"),
            1
        ).otherwise(0)
    ).alias("takeaway_mismatches"),

    F.sum(
        F.when(
            F.col("blocked") != F.col("skater_blocked"),
            1
        ).otherwise(0)
    ).alias("blocked_shot_mismatches"),

    F.sum(
        F.when(
            F.col("opponent_shots") != F.col("shots_against"),
            1
        ).otherwise(0)
    ).alias("goalie_shot_mismatches")
)

display(metric_reconciliation)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 41, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6da45c22-9dca-41ab-8c87-0933a876dbd2)

These mismatches show that team-, skater- and goalie-level files do not always use identical aggregation rules.

##### Profile missing analytical metrics by season

In [40]:
season_missing_profile_df = (
    gold_final_df
    .groupBy("season")
    .agg(
        F.count("*").alias("team_game_rows"),

        F.sum(F.col("hits").isNull().cast("int"))
            .alias("missing_hits"),

        F.sum(F.col("penalty_minutes").isNull().cast("int"))
            .alias("missing_penalty_minutes"),

        F.sum(F.col("power_play_opportunities").isNull().cast("int"))
            .alias("missing_power_play_opportunities"),

        F.sum(F.col("power_play_goals").isNull().cast("int"))
            .alias("missing_power_play_goals"),

        F.sum(F.col("face_off_win_percentage").isNull().cast("int"))
            .alias("missing_faceoff_percentage"),

        F.sum(F.col("giveaways").isNull().cast("int"))
            .alias("missing_giveaways"),

        F.sum(F.col("takeaways").isNull().cast("int"))
            .alias("missing_takeaways"),

        F.sum(F.col("blocked").isNull().cast("int"))
            .alias("missing_blocked"),

        F.sum(F.col("team_save_percentage").isNull().cast("int"))
            .alias("missing_save_percentage"),

        F.sum(F.col("goal_scorers").isNull().cast("int"))
            .alias("missing_goal_scorers"),

        F.sum(F.col("point_contributors").isNull().cast("int"))
            .alias("missing_point_contributors")
    )
    .orderBy("season")
)

display(season_missing_profile_df)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 42, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4aa89f76-72f2-430c-b5ad-bfe521ad1e28)

Hits are unavailable for 2000–01 and 2001–02.

Face_off_win_percentage is unavailable through 2009–10.

Giveaways, takeaways, and blocked are unavailable for 2000–01 and 2001–02.

Goalie save percentage is missing for only the seven previously identified records.

Player contribution measures are complete across all seasons.

Therefore, the common comparison period for the complete set of win factors should begin with 2010–11.

Every season should still be retained in Gold.

##### Add analysis-scope flags

In [41]:
gold_final_df = (
    gold_final_df

    # Physical and puck-management metrics are complete from 2010–11
    .withColumn(
        "complete_metric_period_flag",
        F.when(F.col("season") >= 20102011, 1).otherwise(0)
    )

    # Recommended population for comparing all win factors
    .withColumn(
        "complete_win_factor_analysis_flag",
        F.when(
            (F.col("season") >= 20102011) &
            (F.col("win_loss_analysis_flag") == 1) &
            (F.col("team_save_percentage").isNotNull()),
            1
        ).otherwise(0)
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 43, Finished, Available, Finished, False)

In [42]:
# Validate the scope

display(
    gold_final_df.agg(
        F.count("*").alias("all_team_game_rows"),

        F.sum("win_loss_analysis_flag")
            .alias("win_loss_team_game_rows"),

        F.sum("complete_metric_period_flag")
            .alias("complete_period_team_game_rows"),

        F.sum("complete_win_factor_analysis_flag")
            .alias("complete_win_factor_rows")
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 44, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, de4a9534-2717-4803-ac47-fb5e51778309)

Full Gold history: 47,432 team-game rows

Win/loss comparison: 46,176 rows

Complete-factor period from 2010–11: 25,292 rows

No missing save percentages in the complete-factor period

#### Check game-settlement categories

In [43]:
display(
    gold_final_df
    .groupBy("settled_in")
    .agg(
        F.count("*").alias("team_game_rows"),
        F.countDistinct("game_id").alias("distinct_games")
    )
    .orderBy("settled_in")
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 45, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5811a6db-f3a9-4373-88e2-fafe88eb8601)

In [44]:
# Validate the home/away categories

display(
    gold_final_df
    .groupBy("home_or_away")
    .agg(
        F.count("*").alias("team_game_rows"),
        F.countDistinct("game_id").alias("distinct_games")
    )
    .orderBy("home_or_away")
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 46, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c31e478e-e692-4747-ad69-37394d831d9f)

##### Curate the final Gold dataframe

In [45]:
gold_final_df = (
    gold_final_df
    .withColumn(
        "overtime_flag",
        F.when(F.col("settled_in") == "OT", 1).otherwise(0)
    )
    .withColumn(
        "regulation_flag",
        F.when(F.col("settled_in") == "REG", 1).otherwise(0)
    )
    .withColumn(
        "penalty_kill_percentage",
        F.when(
            F.col("opponent_power_play_opportunities") > 0,
            F.round(
                (
                    1 -
                    F.col("opponent_power_play_goals") /
                    F.col("opponent_power_play_opportunities")
                ) * 100,
                2
            )
        )
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 47, Finished, Available, Finished, False)

In [46]:
gold_nhl_win_factors_df = gold_final_df.select(
    # Keys and game context
    "game_id",
    "season",
    "game_type",
    "game_date",
    "game_date_time_gmt",
    "team_id",
    "team_name",
    "team_abbreviation",
    "opponent_team_id",
    "opponent_team_name",
    "opponent_abbreviation",
    "home_or_away",
    "home_flag",
    "playoff_flag",
    "settled_in",
    "regulation_flag",
    "overtime_flag",
    "head_coach",
    "start_rink_side",

    # Outcome
    "is_win",
    "win_flag",
    "result_category",
    "winning_rows_in_game",
    "win_loss_analysis_flag",

    # Scoring and shooting
    "goals",
    "opponent_goals",
    "goal_difference",
    "shots",
    "opponent_shots",
    "shot_difference",
    "shot_share_percentage",
    "shooting_percentage",

    # Special teams
    "power_play_opportunities",
    "power_play_goals",
    "power_play_efficiency",
    "opponent_power_play_opportunities",
    "opponent_power_play_goals",
    "penalty_kill_percentage",

    # Physical play and possession
    "hits",
    "opponent_hits",
    "hit_difference",
    "penalty_minutes",
    "opponent_penalty_minutes",
    "penalty_minutes_difference",
    "face_off_win_percentage",
    "opponent_face_off_win_percentage",
    "faceoff_percentage_difference",
    "giveaways",
    "opponent_giveaways",
    "giveaway_difference",
    "takeaways",
    "opponent_takeaways",
    "takeaway_difference",
    "puck_management_balance",
    "blocked",
    "opponent_blocked",
    "blocked_shot_difference",

    # Skater-derived factors
    "skaters_used",
    "forwards_used",
    "defencemen_used",
    "skater_assists",
    "team_plus_minus",
    "goal_scorers",
    "point_contributors",
    "top_player_points",
    "points_per_scorer",
    "average_skater_toi_seconds",

    # Goalie-derived factors
    "goalies_used",
    "goalie_saves",
    "team_save_percentage",
    "used_multiple_goalies",
    "total_goalie_toi_seconds",

    # Analysis-scope flags
    "complete_metric_period_flag",
    "complete_win_factor_analysis_flag"
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 48, Finished, Available, Finished, False)

In [47]:
# Validate the curated dataframe

print(f"Rows: {gold_nhl_win_factors_df.count():,}")
print(f"Columns: {len(gold_nhl_win_factors_df.columns)}")

display(gold_nhl_win_factors_df.limit(10))

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 49, Finished, Available, Finished, False)

Rows: 47,432
Columns: 74


SynapseWidget(Synapse.DataFrame, a9ccce13-c695-4f47-a31e-5c3f2fe96553)

##### Run final business-rule tests

In [48]:
business_rule_validation = gold_nhl_win_factors_df.agg(
    F.sum(
        F.when(F.col("goals") < 0, 1).otherwise(0)
    ).alias("negative_goals"),

    F.sum(
        F.when(F.col("shots") < 0, 1).otherwise(0)
    ).alias("negative_shots"),

    F.sum(
        F.when(
            F.col("power_play_goals") >
            F.col("power_play_opportunities"),
            1
        ).otherwise(0)
    ).alias("pp_goals_above_opportunities"),

    F.sum(
        F.when(
            F.col("shooting_percentage").isNotNull() &
            ~F.col("shooting_percentage").between(0, 100),
            1
        ).otherwise(0)
    ).alias("invalid_shooting_percentage"),

    F.sum(
        F.when(
            F.col("power_play_efficiency").isNotNull() &
            ~F.col("power_play_efficiency").between(0, 100),
            1
        ).otherwise(0)
    ).alias("invalid_power_play_efficiency"),

    F.sum(
        F.when(
            F.col("penalty_kill_percentage").isNotNull() &
            ~F.col("penalty_kill_percentage").between(0, 100),
            1
        ).otherwise(0)
    ).alias("invalid_penalty_kill_percentage"),

    F.sum(
        F.when(
            F.col("team_save_percentage").isNotNull() &
            ~F.col("team_save_percentage").between(0, 100),
            1
        ).otherwise(0)
    ).alias("invalid_save_percentage"),

    F.sum(
        F.when(
            ~F.col("home_flag").isin(0, 1),
            1
        ).otherwise(0)
    ).alias("invalid_home_flags"),

    F.sum(
        F.when(
            ~F.col("win_flag").isin(0, 1),
            1
        ).otherwise(0)
    ).alias("invalid_win_flags")
)

display(business_rule_validation)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 50, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 37874dab-0ccc-4b52-8efa-8e0052fb783d)

The Gold dataframe has passed:

Grain and uniqueness tests

Referential-integrity tests

Outcome-balance tests

Missing-data profiling

Business-rule validation

##### Save the Gold Delta table

In [49]:
(
    gold_nhl_win_factors_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "nhl_lakehouse_gold.dbo.gold_nhl_win_factors"
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 51, Finished, Available, Finished, False)

In [50]:
# Verify the saved table

saved_gold_df = spark.table("nhl_lakehouse_gold.dbo.gold_nhl_win_factors")

print(f"Saved rows: {saved_gold_df.count():,}")
print(f"Saved columns: {len(saved_gold_df.columns)}")

display(
    saved_gold_df.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("game_id").alias("distinct_games"),
        F.countDistinct(
            F.struct("game_id", "team_id")
        ).alias("distinct_team_games")
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 52, Finished, Available, Finished, False)

Saved rows: 47,432
Saved columns: 74


SynapseWidget(Synapse.DataFrame, 19e9b8e4-60e6-4e2c-a812-43f14c28d2cb)

##### Create the Team dimension

In [51]:
gold_dim_team_df = (
    gold_nhl_win_factors_df
    .select(
        "team_id",
        "team_name",
        "team_abbreviation"
    )
    .distinct()
    .orderBy("team_id")
)

display(gold_dim_team_df)

print(f"Team dimension rows: {gold_dim_team_df.count()}")

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 53, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f3b4c21b-423c-488b-9ce0-8efa6b1b894c)

Team dimension rows: 33


In [52]:
# Validate uniqueness

display(
    gold_dim_team_df.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("team_id").alias("distinct_team_ids")
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 54, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bf088216-f480-4d60-8e0e-7101ecbeea2f)

##### Create the Opponent dimension

In [53]:
gold_dim_opponent_df = (
    gold_nhl_win_factors_df
    .select(
        "opponent_team_id",
        "opponent_team_name",
        "opponent_abbreviation"
    )
    .distinct()
    .orderBy("opponent_team_id")
)

display(gold_dim_opponent_df)

print(f"Opponent dimension rows: {gold_dim_opponent_df.count()}")

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 55, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, adeacf32-aad1-446b-bd4f-062671a314d4)

Opponent dimension rows: 33


In [54]:
# Validate its key

display(
    gold_dim_opponent_df.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("opponent_team_id")
            .alias("distinct_opponent_team_ids"),
        F.sum(
            F.when(
                F.col("opponent_team_id").isNull(),
                1
            ).otherwise(0)
        ).alias("null_opponent_ids")
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 56, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ce08d914-31ab-421b-b642-664e3707f22f)

##### Create a continuous Date dimension

In [55]:
date_bounds = (
    gold_nhl_win_factors_df
    .agg(
        F.min("game_date").alias("minimum_date"),
        F.max("game_date").alias("maximum_date")
    )
    .first()
)

minimum_date = date_bounds["minimum_date"]
maximum_date = date_bounds["maximum_date"]

print(f"Minimum game date: {minimum_date}")
print(f"Maximum game date: {maximum_date}")

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 57, Finished, Available, Finished, False)

Minimum game date: 2000-10-04
Maximum game date: 2020-09-29


In [56]:
# Create the calendar

gold_dim_date_df = (
    spark.sql(
        f"""
        SELECT EXPLODE(
            SEQUENCE(
                TO_DATE('{minimum_date}'),
                TO_DATE('{maximum_date}'),
                INTERVAL 1 DAY
            )
        ) AS game_date
        """
    )
    .withColumn(
        "date_key",
        F.date_format("game_date", "yyyyMMdd").cast("int")
    )
    .withColumn("calendar_year", F.year("game_date"))
    .withColumn("quarter_number", F.quarter("game_date"))
    .withColumn(
        "quarter_name",
        F.concat(
            F.lit("Q"),
            F.quarter("game_date")
        )
    )
    .withColumn("month_number", F.month("game_date"))
    .withColumn("month_name", F.date_format("game_date", "MMMM"))
    .withColumn("month_short_name", F.date_format("game_date", "MMM"))
    .withColumn("year_month", F.date_format("game_date", "yyyy-MM"))
    .withColumn("day_of_month", F.dayofmonth("game_date"))
    .withColumn("day_of_week_number", F.dayofweek("game_date"))
    .withColumn("day_name", F.date_format("game_date", "EEEE"))
    .withColumn(
        "is_weekend",
        F.when(F.dayofweek("game_date").isin(1, 7), 1).otherwise(0)
    )
    .select(
        "date_key",
        "game_date",
        "calendar_year",
        "quarter_number",
        "quarter_name",
        "month_number",
        "month_name",
        "month_short_name",
        "year_month",
        "day_of_month",
        "day_of_week_number",
        "day_name",
        "is_weekend"
    )
    .orderBy("game_date")
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 58, Finished, Available, Finished, False)

In [57]:
# Validate the calendar

display(
    gold_dim_date_df.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("date_key").alias("distinct_date_keys"),
        F.countDistinct("game_date").alias("distinct_dates"),
        F.min("game_date").alias("minimum_date"),
        F.max("game_date").alias("maximum_date")
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 59, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dac891ea-567f-4f02-99f7-589bab84f5f7)

##### Create the Season dimension

In [58]:
gold_dim_season_df = (
    gold_nhl_win_factors_df
    .select("season")
    .distinct()
    .withColumn(
        "season_start_year",
        F.substring(
            F.col("season").cast("string"),
            1,
            4
        ).cast("int")
    )
    .withColumn(
        "season_end_year",
        F.substring(
            F.col("season").cast("string"),
            5,
            4
        ).cast("int")
    )
    .withColumn(
        "season_label",
        F.concat(
            F.col("season_start_year").cast("string"),
            F.lit("-"),
            F.substring(
                F.col("season_end_year").cast("string"),
                3,
                2
            )
        )
    )
    .withColumn(
        "complete_metric_period_flag",
        F.when(F.col("season") >= 20102011, 1).otherwise(0)
    )
    .select(
        "season",
        "season_label",
        "season_start_year",
        "season_end_year",
        "complete_metric_period_flag"
    )
    .orderBy("season")
)

display(gold_dim_season_df)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 60, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ea7d34f3-2782-4454-94b9-e18bb7b1791c)

In [59]:
# Validate it

display(
    gold_dim_season_df.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("season").alias("distinct_seasons"),
        F.min("season_start_year").alias("earliest_start_year"),
        F.max("season_end_year").alias("latest_end_year")
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 61, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d31b7f02-55b7-473e-b705-f7a5a2c91183)

The missing 2004–05 season is expected because the NHL season was cancelled due to a 310-day labor lockout led by NHL Commissioner Gary Bettman and the owners over a salary cap dispute.

##### Create the central Fact table

In [60]:
gold_fact_team_game_df = (
    gold_nhl_win_factors_df
    .withColumn(
        "date_key",
        F.date_format("game_date", "yyyyMMdd").cast("int")
    )
    .drop(
        "game_date",
        "team_name",
        "team_abbreviation",
        "opponent_team_name",
        "opponent_abbreviation",
        "is_win",
        "winning_rows_in_game",
        "complete_metric_period_flag"
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 62, Finished, Available, Finished, False)

In [61]:
# Check its size and grain

print(f"Fact rows: {gold_fact_team_game_df.count():,}")
print(f"Fact columns: {len(gold_fact_team_game_df.columns)}")

display(
    gold_fact_team_game_df.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("game_id").alias("distinct_games"),
        F.countDistinct(
            F.struct("game_id", "team_id")
        ).alias("distinct_team_games")
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 63, Finished, Available, Finished, False)

Fact rows: 47,432
Fact columns: 67


SynapseWidget(Synapse.DataFrame, 2107a72a-a135-4fe2-b5f0-2eee1aa2277c)

In [62]:
# Validate all four foreign keys

foreign_key_validation = [
    (
        "team_id",
        gold_fact_team_game_df.select("team_id").distinct()
        .join(
            gold_dim_team_df.select("team_id"),
            on="team_id",
            how="left_anti"
        ).count()
    ),
    (
        "opponent_team_id",
        gold_fact_team_game_df.select("opponent_team_id").distinct()
        .join(
            gold_dim_opponent_df.select("opponent_team_id"),
            on="opponent_team_id",
            how="left_anti"
        ).count()
    ),
    (
        "date_key",
        gold_fact_team_game_df.select("date_key").distinct()
        .join(
            gold_dim_date_df.select("date_key"),
            on="date_key",
            how="left_anti"
        ).count()
    ),
    (
        "season",
        gold_fact_team_game_df.select("season").distinct()
        .join(
            gold_dim_season_df.select("season"),
            on="season",
            how="left_anti"
        ).count()
    )
]

display(
    spark.createDataFrame(
        foreign_key_validation,
        ["foreign_key", "unmatched_key_count"]
    )
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 64, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 671e9acf-c0ad-4a11-9510-f17141ad4a7f)

##### Save the five star-schema tables

In [63]:
star_schema_tables = {
    "gold_fact_team_game": gold_fact_team_game_df,
    "gold_dim_team": gold_dim_team_df,
    "gold_dim_opponent": gold_dim_opponent_df,
    "gold_dim_date": gold_dim_date_df,
    "gold_dim_season": gold_dim_season_df
}

for table_name, dataframe in star_schema_tables.items():

    full_table_name = f"nhl_lakehouse_gold.dbo.{table_name}"

    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )

    print(f"Saved {full_table_name}: {dataframe.count():,} rows")

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 65, Finished, Available, Finished, False)

Saved nhl_lakehouse_gold.dbo.gold_fact_team_game: 47,432 rows
Saved nhl_lakehouse_gold.dbo.gold_dim_team: 33 rows
Saved nhl_lakehouse_gold.dbo.gold_dim_opponent: 33 rows
Saved nhl_lakehouse_gold.dbo.gold_dim_date: 7,301 rows
Saved nhl_lakehouse_gold.dbo.gold_dim_season: 19 rows


##### Create the complete analysis population

In [64]:
fact_df = spark.table("nhl_lakehouse_gold.dbo.gold_fact_team_game")

win_factor_analysis_df = (
    fact_df
    .filter(F.col("complete_win_factor_analysis_flag") == 1)
)

display(
    win_factor_analysis_df
    .groupBy("result_category")
    .agg(
        F.count("*").alias("team_game_rows"),
        F.countDistinct("game_id").alias("distinct_games")
    )
    .orderBy("result_category")
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 66, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e29a48bd-8c5a-4450-a733-5114015b79ac)

##### Analysis 1: Winner-versus-loser averages

In [65]:
comparison_metrics = {
    "Home-team share": "home_flag",
    "Shot difference": "shot_difference",
    "Shot share percentage": "shot_share_percentage",
    "Shooting percentage": "shooting_percentage",
    "Power-play efficiency": "power_play_efficiency",
    "Penalty-kill percentage": "penalty_kill_percentage",
    "Save percentage": "team_save_percentage",
    "Faceoff win percentage": "face_off_win_percentage",
    "Hits": "hits",
    "Penalty minutes": "penalty_minutes",
    "Giveaways": "giveaways",
    "Takeaways": "takeaways",
    "Blocked shots": "blocked",
    "Goal scorers": "goal_scorers",
    "Point contributors": "point_contributors",
    "Top player points": "top_player_points"
}

average_expressions = [
    F.avg(column_name).alias(column_name)
    for column_name in comparison_metrics.values()
]

average_by_result_df = (
    win_factor_analysis_df
    .groupBy("result_category")
    .agg(*average_expressions)
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 67, Finished, Available, Finished, False)

In [66]:
comparison_rows = []

for metric_label, column_name in comparison_metrics.items():
    metric_result = (
        average_by_result_df
        .select(
            "result_category",
            F.col(column_name).alias("average_value")
        )
        .withColumn("metric", F.lit(metric_label))
    )

    comparison_rows.append(metric_result)

winner_loser_long_df = comparison_rows[0]

for dataframe in comparison_rows[1:]:
    winner_loser_long_df = winner_loser_long_df.unionByName(dataframe)

winner_loser_comparison_df = (
    winner_loser_long_df
    .groupBy("metric")
    .pivot("result_category", ["Win", "Loss"])
    .agg(F.first("average_value"))
    .withColumn(
        "winner_minus_loser",
        F.col("Win") - F.col("Loss")
    )
    .withColumn(
        "relative_difference_percentage",
        F.when(
            F.col("Loss") != 0,
            (F.col("Win") - F.col("Loss")) /
            F.col("Loss") * 100
        )
    )
    .select(
        "metric",
        F.round("Win", 2).alias("winner_average"),
        F.round("Loss", 2).alias("loser_average"),
        F.round("winner_minus_loser", 2).alias("difference"),
        F.round(
            "relative_difference_percentage",
            2
        ).alias("relative_difference_percentage")
    )
    .orderBy(F.desc(F.abs(F.col("relative_difference_percentage"))))
)

display(winner_loser_comparison_df)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 68, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 78311de7-8131-4614-ac5d-99014518b27c)

In [67]:
winner_loser_comparison_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nhl_lakehouse_gold.dbo.gold_analysis_01_winner_loser_averages")

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 69, Finished, Available, Finished, False)

##### Analysis 2: Correlation with Winning

In [68]:
win_factors_df = spark.read.table(
    "nhl_lakehouse_gold.dbo.gold_nhl_win_factors"
)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 70, Finished, Available, Finished, False)

In [69]:
# Keep valid winner/loser rows with complete win-factor metrics
analysis_2_base_df = (
    win_factors_df
    .filter(
        (F.col("win_loss_analysis_flag") == 1) &
        (F.col("complete_win_factor_analysis_flag") == 1)
    )
)

# Metrics used in Analysis 1
metrics = [
    ("shot_difference", "Shot differential"),
    ("shooting_percentage", "Shooting percentage"),
    ("goal_scorers", "Goal scorers"),
    ("power_play_efficiency", "Power-play efficiency"),
    ("point_contributors", "Point contributors"),
    ("top_player_points", "Top-player points"),
    ("home_flag", "Home-team status"),
    ("blocked", "Blocked shots"),
    ("penalty_kill_percentage", "Penalty-kill percentage"),
    ("takeaways", "Takeaways"),
    ("team_save_percentage", "Save percentage"),
    ("penalty_minutes", "Penalty minutes"),
    ("hits", "Hits"),
    ("giveaways", "Giveaways"),
    ("face_off_win_percentage", "Faceoff win percentage"),
    ("shot_share_percentage", "Shot-share percentage")
]

# Calculate correlation between each metric and win_flag
correlation_rows = []

for column_name, metric_name in metrics:
    correlation_value = (
        analysis_2_base_df
        .select(F.corr(F.col(column_name), F.col("win_flag")).alias("correlation"))
        .first()["correlation"]
    )

    correlation_rows.append(
        (metric_name, column_name, correlation_value)
    )

# Create the Analysis 2 result
analysis_2_df = spark.createDataFrame(
    correlation_rows,
    ["metric", "column_name", "correlation_with_win"]
).withColumn(
    "correlation_with_win",
    F.round("correlation_with_win", 4)
).withColumn(
    "absolute_correlation",
    F.round(F.abs("correlation_with_win"), 4)
).orderBy(
    F.desc("absolute_correlation")
)

display(analysis_2_df)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 71, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9d5af6b0-9347-4c45-8eaa-58828b2511f6)

In [70]:
analysis_2_final_df = (
    analysis_2_df
    .withColumn(
        "direction",
        F.when(F.col("correlation_with_win") > 0, "Positive")
         .when(F.col("correlation_with_win") < 0, "Negative")
         .otherwise("None")
    )
    .withColumn(
        "relationship_strength",
        F.when(F.col("absolute_correlation") >= 0.50, "Strong")
         .when(F.col("absolute_correlation") >= 0.30, "Moderate")
         .when(F.col("absolute_correlation") >= 0.10, "Weak")
         .otherwise("Very weak")
    )
)

analysis_2_final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nhl_lakehouse_gold.dbo.gold_analysis_02_win_factor_correlations")

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 72, Finished, Available, Finished, False)

##### Analysis 3: Win rate by performance level

In [71]:
from pyspark.sql.window import Window

analysis_3_base_df = (
    win_factors_df
    .filter(
        (F.col("win_loss_analysis_flag") == 1) &
        (F.col("complete_win_factor_analysis_flag") == 1) &
        F.col("shooting_percentage").isNotNull()
    )
)

shooting_band_window = Window.orderBy("shooting_percentage")

shooting_band_df = (
    analysis_3_base_df
    .withColumn(
        "performance_band_number",
        F.ntile(4).over(shooting_band_window)
    )
    .withColumn(
        "performance_band",
        F.when(F.col("performance_band_number") == 1, "Q1 – Lowest")
         .when(F.col("performance_band_number") == 2, "Q2 – Low")
         .when(F.col("performance_band_number") == 3, "Q3 – High")
         .otherwise("Q4 – Highest")
    )
    .groupBy("performance_band_number", "performance_band")
    .agg(
        F.count("*").alias("team_game_rows"),
        F.round(F.min("shooting_percentage"), 2).alias("minimum_value"),
        F.round(F.max("shooting_percentage"), 2).alias("maximum_value"),
        F.round(F.avg("shooting_percentage"), 2).alias("average_value"),
        F.round(F.avg("win_flag") * 100, 2).alias("win_percentage")
    )
    .orderBy("performance_band_number")
)

display(shooting_band_df)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 73, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b535fa19-bf2a-44eb-a82d-b8c5dd774f99)

Quartiles contain approximately equal row counts. Tied or rounded metric values may appear in adjacent bands.

In [72]:
from functools import reduce

band_metrics = [
    ("shooting_percentage", "Shooting percentage"),
    ("team_save_percentage", "Save percentage"),
    ("power_play_efficiency", "Power-play efficiency"),
    ("penalty_kill_percentage", "Penalty-kill percentage"),
    ("blocked", "Blocked shots"),
    ("takeaways", "Takeaways"),
    ("hits", "Hits"),
    ("penalty_minutes", "Penalty minutes")
]

def create_performance_bands(column_name, metric_name):
    metric_window = Window.orderBy(F.col(column_name))

    return (
        analysis_3_base_df
        .filter(F.col(column_name).isNotNull())
        .withColumn(
            "performance_band_number",
            F.ntile(4).over(metric_window)
        )
        .withColumn(
            "performance_band",
            F.when(F.col("performance_band_number") == 1, "Q1 – Lowest")
             .when(F.col("performance_band_number") == 2, "Q2 – Low")
             .when(F.col("performance_band_number") == 3, "Q3 – High")
             .otherwise("Q4 – Highest")
        )
        .groupBy("performance_band_number", "performance_band")
        .agg(
            F.count("*").alias("team_game_rows"),
            F.round(F.min(column_name), 2).alias("minimum_value"),
            F.round(F.max(column_name), 2).alias("maximum_value"),
            F.round(F.avg(column_name), 2).alias("average_value"),
            F.round(F.avg("win_flag") * 100, 2).alias("win_percentage")
        )
        .withColumn("metric", F.lit(metric_name))
        .withColumn("column_name", F.lit(column_name))
        .select(
            "metric",
            "column_name",
            "performance_band_number",
            "performance_band",
            "team_game_rows",
            "minimum_value",
            "maximum_value",
            "average_value",
            "win_percentage"
        )
    )

band_results = [
    create_performance_bands(column_name, metric_name)
    for column_name, metric_name in band_metrics
]

analysis_3_df = reduce(
    lambda first_df, next_df: first_df.unionByName(next_df),
    band_results
).orderBy("metric", "performance_band_number")

display(analysis_3_df)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 74, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8e051ed4-7f45-432a-b4a4-88da57e94575)

In [73]:
analysis_3_final_df = (
    analysis_3_df
    .withColumn(
        "performance_band",
        F.when(F.col("performance_band_number") == 1, "Q1 - Lowest")
         .when(F.col("performance_band_number") == 2, "Q2 - Low")
         .when(F.col("performance_band_number") == 3, "Q3 - High")
         .otherwise("Q4 - Highest")
    )
)

display(analysis_3_final_df)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 75, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fd4a98cc-d94a-4ee8-85fd-7ab78c3b013e)

In [74]:
analysis_3_final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nhl_lakehouse_gold.dbo.gold_analysis_03_performance_bands")

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 76, Finished, Available, Finished, False)

##### Analysis 4: Do winning factors differ between regular-season and playoff games?

In [75]:
analysis_4_base_df = (
    win_factors_df
    .filter(
        (F.col("win_loss_analysis_flag") == 1) &
        (F.col("complete_win_factor_analysis_flag") == 1)
    )
)

analysis_4_counts_df = (
    analysis_4_base_df
    .withColumn(
        "game_stage",
        F.when(F.col("playoff_flag") == 1, "Playoffs")
         .otherwise("Regular season")
    )
    .groupBy("game_stage", "win_flag")
    .agg(
        F.count("*").alias("team_game_rows"),
        F.countDistinct("game_id").alias("distinct_games")
    )
    .orderBy("game_stage", F.desc("win_flag"))
)

display(analysis_4_counts_df)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 77, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ec6615f8-273f-4650-bfe8-3b21ec81e43b)

In [76]:
analysis_4_metrics = [
    ("shooting_percentage", "Shooting percentage"),
    ("team_save_percentage", "Save percentage"),
    ("power_play_efficiency", "Power-play efficiency"),
    ("penalty_kill_percentage", "Penalty-kill percentage"),
    ("blocked", "Blocked shots"),
    ("takeaways", "Takeaways"),
    ("hits", "Hits"),
    ("penalty_minutes", "Penalty minutes")
]

analysis_4_stage_df = (
    analysis_4_base_df
    .withColumn(
        "game_stage",
        F.when(F.col("playoff_flag") == 1, "Playoffs")
         .otherwise("Regular season")
    )
)

def compare_by_game_stage(column_name, metric_name):
    return (
        analysis_4_stage_df
        .groupBy("game_stage")
        .pivot("win_flag", [0, 1])
        .agg(F.avg(column_name))
        .withColumnRenamed("0", "loser_average")
        .withColumnRenamed("1", "winner_average")
        .withColumn("metric", F.lit(metric_name))
        .withColumn("column_name", F.lit(column_name))
        .withColumn(
            "difference",
            F.col("winner_average") - F.col("loser_average")
        )
        .select(
            "game_stage",
            "metric",
            "column_name",
            F.round("winner_average", 2).alias("winner_average"),
            F.round("loser_average", 2).alias("loser_average"),
            F.round("difference", 2).alias("difference")
        )
    )

stage_results = [
    compare_by_game_stage(column_name, metric_name)
    for column_name, metric_name in analysis_4_metrics
]

analysis_4_df = reduce(
    lambda first_df, next_df: first_df.unionByName(next_df),
    stage_results
).orderBy("metric", "game_stage")

display(analysis_4_df)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 78, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, eafa7045-4a7f-44f5-9ffb-a34f3acc52b1)

In [77]:
analysis_4_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nhl_lakehouse_gold.dbo.gold_analysis_04_regular_vs_playoffs")

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 79, Finished, Available, Finished, False)

##### Analysis 5: Have the key winning factors changed across seasons?

In [78]:
analysis_5_base_df = (
    win_factors_df
    .filter(
        (F.col("win_loss_analysis_flag") == 1) &
        (F.col("complete_win_factor_analysis_flag") == 1)
    )
)

analysis_5_season_counts_df = (
    analysis_5_base_df
    .groupBy("season")
    .agg(
        F.count("*").alias("team_game_rows"),
        F.countDistinct("game_id").alias("distinct_games"),
        F.sum("win_flag").alias("winning_rows"),
        (F.count("*") - F.sum("win_flag")).alias("losing_rows")
    )
    .orderBy("season")
)

display(analysis_5_season_counts_df)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 80, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5328af7a-98bf-490b-97e3-3411afd300dc)

In [79]:
analysis_5_metrics = [
    ("shooting_percentage", "Shooting percentage"),
    ("team_save_percentage", "Save percentage"),
    ("power_play_efficiency", "Power-play efficiency"),
    ("penalty_kill_percentage", "Penalty-kill percentage"),
    ("blocked", "Blocked shots"),
    ("takeaways", "Takeaways"),
    ("hits", "Hits"),
    ("penalty_minutes", "Penalty minutes")
]

def compare_by_season(column_name, metric_name):
    return (
        analysis_5_base_df
        .groupBy("season")
        .pivot("win_flag", [0, 1])
        .agg(F.avg(column_name))
        .withColumnRenamed("0", "loser_average")
        .withColumnRenamed("1", "winner_average")
        .withColumn("metric", F.lit(metric_name))
        .withColumn("column_name", F.lit(column_name))
        .withColumn(
            "difference",
            F.col("winner_average") - F.col("loser_average")
        )
        .withColumn(
            "season_label",
            F.concat(
                F.substring(F.col("season").cast("string"), 1, 4),
                F.lit("-"),
                F.substring(F.col("season").cast("string"), 7, 2)
            )
        )
        .select(
            "season",
            "season_label",
            "metric",
            "column_name",
            F.round("winner_average", 2).alias("winner_average"),
            F.round("loser_average", 2).alias("loser_average"),
            F.round("difference", 2).alias("difference")
        )
    )

season_results = [
    compare_by_season(column_name, metric_name)
    for column_name, metric_name in analysis_5_metrics
]

analysis_5_df = reduce(
    lambda first_df, next_df: first_df.unionByName(next_df),
    season_results
).orderBy("metric", "season")

display(analysis_5_df)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 81, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bd5f4315-2027-47d4-a8ac-96ffd450ce7a)

In [80]:
analysis_5_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nhl_lakehouse_gold.dbo.gold_analysis_05_season_trends")

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 82, Finished, Available, Finished, False)

##### Validate the tables

In [81]:
analysis_tables = [
    ("nhl_lakehouse_gold.dbo.gold_analysis_01_winner_loser_averages", 16),
    ("nhl_lakehouse_gold.dbo.gold_analysis_02_win_factor_correlations", 16),
    ("nhl_lakehouse_gold.dbo.gold_analysis_03_performance_bands", 32),
    ("nhl_lakehouse_gold.dbo.gold_analysis_04_regular_vs_playoffs", 16),
    ("nhl_lakehouse_gold.dbo.gold_analysis_05_season_trends", 80)
]

validation_results = []

for table_name, expected_rows in analysis_tables:
    actual_rows = spark.table(table_name).count()

    validation_results.append(
        (
            table_name,
            expected_rows,
            actual_rows,
            "PASS" if actual_rows == expected_rows else "FAIL"
        )
    )

validation_df = spark.createDataFrame(
    validation_results,
    ["table_name", "expected_rows", "actual_rows", "validation_status"]
)

display(validation_df)

StatementMeta(, fa0bec33-0d9e-4ec1-8abb-dba0dc762048, 83, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, db6e394d-aeba-42a2-aba6-2720a9ff032c)